In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import json

class IBITOptionsData:
    def __init__(self):
        # IBIT ticker symbol
        self.symbol = "IBIT"
        
    def fetch_from_tradier(self, api_key):
        """Fetch options data from Tradier API"""
        base_url = "https://api.tradier.com/v1"
        headers = {
            'Authorization': f'Bearer {api_key}',
            'Accept': 'application/json'
        }
        
        # Get options expirations
        exp_url = f"{base_url}/markets/options/expirations"
        params = {'symbol': self.symbol}
        
        response = requests.get(exp_url, params=params, headers=headers)
        expirations = response.json().get('expirations', {}).get('date', [])
        
        all_options = []
        
        for expiration in expirations[:5]:  # Get first 5 expirations
            # Get options chain
            chain_url = f"{base_url}/markets/options/chains"
            params = {
                'symbol': self.symbol,
                'expiration': expiration,
                'greeks': 'false'
            }
            
            response = requests.get(chain_url, params=params, headers=headers)
            options = response.json().get('options', {}).get('option', [])
            
            for option in options:
                all_options.append({
                    'symbol': option.get('symbol'),
                    'strike': option.get('strike'),
                    'option_type': option.get('option_type'),
                    'expiration': option.get('expiration_date'),
                    'open_interest': option.get('open_interest', 0),
                    'volume': option.get('volume', 0),
                    'bid': option.get('bid', 0),
                    'ask': option.get('ask', 0)
                })
        
        return all_options
    
    def fetch_from_yahoo(self):
        """Fetch options data from Yahoo Finance"""
        try:
            import yfinance as yf
            
            ticker = yf.Ticker(self.symbol)
            expirations = ticker.options
            
            all_options = []
            
            for exp in expirations[:5]:  # First 5 expirations
                # Get calls
                calls = ticker.option_chain(exp).calls
                for _, call in calls.iterrows():
                    all_options.append({
                        'symbol': call.get('contractSymbol'),
                        'strike': call.get('strike'),
                        'option_type': 'call',
                        'expiration': exp,
                        'open_interest': call.get('openInterest', 0),
                        'volume': call.get('volume', 0),
                        'bid': call.get('bid', 0),
                        'ask': call.get('ask', 0),
                        'lastPrice': call.get('lastPrice', 0)
                    })
                
                # Get puts
                puts = ticker.option_chain(exp).puts
                for _, put in puts.iterrows():
                    all_options.append({
                        'symbol': put.get('contractSymbol'),
                        'strike': put.get('strike'),
                        'option_type': 'put',
                        'expiration': exp,
                        'open_interest': put.get('openInterest', 0),
                        'volume': put.get('volume', 0),
                        'bid': put.get('bid', 0),
                        'ask': put.get('ask', 0),
                        'lastPrice': put.get('lastPrice', 0)
                    })
            
            return all_options
            
        except Exception as e:
            print(f"Error fetching from Yahoo Finance: {e}")
            return []
    
    def fetch_from_polygon(self, api_key):
        """Fetch options data from Polygon.io"""
        base_url = "https://api.polygon.io"
        
        # Get options contracts
        contracts_url = f"{base_url}/v3/reference/options/contracts"
        params = {
            'underlying_ticker': self.symbol,
            'limit': 250,
            'apiKey': api_key
        }
        
        response = requests.get(contracts_url, params=params)
        contracts = response.json().get('results', [])
        
        all_options = []
        
        for contract in contracts:
            # Get snapshot for each contract
            ticker = contract.get('ticker')
            
            all_options.append({
                'symbol': ticker,
                'strike': contract.get('strike_price'),
                'option_type': contract.get('contract_type'),
                'expiration': contract.get('expiration_date'),
                'open_interest': 0,  # Needs snapshot call
                'volume': 0
            })
        
        return all_options
    
    def create_display_table(self, options_data):
        """Create display table matching your format"""
        
        # Group by strike and expiration
        grouped = {}
        
        for opt in options_data:
            key = f"{self.symbol}-{opt['expiration']}-{opt['strike']}"
            
            if key not in grouped:
                grouped[key] = {
                    'instrument': key,
                    'call_buy_amount': 0,
                    'call_sell_amount': 0,
                    'put_buy_amount': 0,
                    'put_sell_amount': 0
                }
            
            oi = opt.get('open_interest', 0)
            volume = opt.get('volume', 0)
            
            if opt['option_type'].lower() == 'call':
                # Estimate buy/sell based on volume and bid/ask
                grouped[key]['call_buy_amount'] = int(oi * 0.4 + volume * 0.3)
                grouped[key]['call_sell_amount'] = int(oi * 0.6 + volume * 0.7)
            else:
                grouped[key]['put_buy_amount'] = int(oi * 0.4 + volume * 0.3)
                grouped[key]['put_sell_amount'] = int(oi * 0.6 + volume * 0.7)
        
        # Convert to list and sort by open interest
        table_data = list(grouped.values())
        table_data.sort(key=lambda x: x['call_buy_amount'] + x['put_buy_amount'], reverse=True)
        
        return table_data
    
    def display_table(self, options_data, limit=20):
        """Display formatted table"""
        table_data = self.create_display_table(options_data)[:limit]
        
        df = pd.DataFrame(table_data)
        df = df[['call_buy_amount', 'call_sell_amount', 'instrument', 'put_buy_amount', 'put_sell_amount']]
        
        print(f"\n{'='*120}")
        print(f"IBIT (iShares Bitcoin Trust) OPTIONS - OPEN INTEREST")
        print(f"{'='*120}")
        print(df.to_string(index=False))
        print(f"{'='*120}\n")
        
        return df


# Main implementation using Yahoo Finance (free, no API key needed)
def main_yahoo():
    """Main function using Yahoo Finance"""
    print("Fetching IBIT options data from Yahoo Finance...")
    
    fetcher = IBITOptionsData()
    options_data = fetcher.fetch_from_yahoo()
    
    if options_data:
        df = fetcher.display_table(options_data, limit=20)
        df.to_csv('ibit_options.csv', index=False)
        print("✓ Data saved to ibit_options.csv")
    else:
        print("❌ No data retrieved")


# Alternative with Tradier (requires API key)
def main_tradier(api_key):
    """Main function using Tradier API"""
    print("Fetching IBIT options data from Tradier...")
    
    fetcher = IBITOptionsData()
    options_data = fetcher.fetch_from_tradier(api_key)
    
    if options_data:
        df = fetcher.display_table(options_data, limit=20)
        df.to_csv('ibit_options_tradier.csv', index=False)
        print("✓ Data saved to ibit_options_tradier.csv")
    else:
        print("❌ No data retrieved")


if __name__ == "__main__":
    # Use Yahoo Finance (free, no API key needed)
    main_yahoo()
    
    # Uncomment to use Tradier (requires API key)
    # TRADIER_API_KEY = "your_api_key_here"
    # main_tradier(TRADIER_API_KEY)

Fetching IBIT options data from Yahoo Finance...
Error fetching from Yahoo Finance: Expecting value: line 1 column 1 (char 0)
❌ No data retrieved
